In [1]:
import pandas as pd
import numpy as np
import joblib
import os

# -------------------------------
# Paths
# -------------------------------
BASE_DIR = os.getcwd()
PROJECT_DIR = os.path.abspath(os.path.join(BASE_DIR, ".."))

data_path = os.path.join(PROJECT_DIR, "data", "student_data.xlsx")
model_dir = os.path.join(PROJECT_DIR, "model")

os.makedirs(model_dir, exist_ok=True)

# -------------------------------
# Load Data
# -------------------------------
df = pd.read_excel(data_path)
df.columns = df.columns.str.strip()

# -------------------------------
# CLEAN TARGET (IMPORTANT)
# -------------------------------
df['Target'] = (
    (df['Study_Hours_per_Week'] > 8) &
    (df['Attendance_Rate'] > 60) &
    (df['Past_Exam_Scores'] > 60) &
    (df['Assignments'] > 15) &
    (df['Internal_Marks'] > 60)
).astype(int)

y = df['Target']

# -------------------------------
# Features
# -------------------------------
X = df[['Study_Hours_per_Week',
        'Attendance_Rate',
        'Past_Exam_Scores',
        'Assignments',
        'Internal_Marks']]

# -------------------------------
# Scaling
# -------------------------------
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# Train-Test Split
# -------------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# -------------------------------
# MODELS (3 MODELS NOW)
# -------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}

best_model = None
best_accuracy = 0
best_name = ""

print("\n🔍 Model Evaluation:\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {acc}")
    print(classification_report(y_test, y_pred))

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_name = name

print("\n🏆 Best Model Selected:", best_name)

# -------------------------------
# Nearest Neighbors
# -------------------------------
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=3)
nn.fit(X_scaled)

# -------------------------------
# Save
# -------------------------------
joblib.dump(best_model, os.path.join(model_dir, "model.pkl"))
joblib.dump(nn, os.path.join(model_dir, "nn.pkl"))
joblib.dump(df, os.path.join(model_dir, "data.pkl"))
joblib.dump(scaler, os.path.join(model_dir, "scaler.pkl"))
joblib.dump(best_name, os.path.join(model_dir, "model_name.pkl"))

print("\n✅ Training complete with multiple models!")


🔍 Model Evaluation:

Logistic Regression Accuracy: 0.8450704225352113
              precision    recall  f1-score   support

           0       0.85      0.80      0.83        66
           1       0.84      0.88      0.86        76

    accuracy                           0.85       142
   macro avg       0.85      0.84      0.84       142
weighted avg       0.85      0.85      0.84       142

Decision Tree Accuracy: 0.9929577464788732
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        66
           1       0.99      1.00      0.99        76

    accuracy                           0.99       142
   macro avg       0.99      0.99      0.99       142
weighted avg       0.99      0.99      0.99       142

Random Forest Accuracy: 0.9929577464788732
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        66
           1       0.99      1.00      0.99        76

    accuracy             